In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)



In [ ]:
import sys

import os
import glob

import numpy as np

from astropy.io import fits

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
def plot():

    '''
    '''

    # Set common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    # Get the corresponding files of the BEAGLE fits
    files = glob.glob(f'{results}/beagle_fits/e24_f775w_dropouts_2csfh_no_lya/*_GOODS*_BEAGLE.fits.gz')

    # For each BEAGLE fit results file
    for i, file in enumerate(files):

        # Check if the file is empty; skip if so
        if os.stat(file).st_size == 0:
            print(f'File {os.path.basename(file)} is empty. Skipping.')
            continue

        # Get the galaxy ID from the file name
        id = os.path.basename(file).split('_BEAGLE')[0]

        # Get the HDUL of the BEAGLE fit results file
        hdul = fits.open(file)

        probs = hdul['POSTERIOR PDF'].data['probability']

        m_tot = hdul['GALAXY PROPERTIES'].data['M_tot']
        cst = 10**hdul['POSTERIOR PDF'].data['current_sfr_timescale']
        msa = 10**hdul['POSTERIOR PDF'].data['max_stellar_age']
        sfr = hdul['STAR FORMATION'].data['SFR']
        tau = 10**hdul['POSTERIOR PDF'].data['tau']

        alpha = np.zeros(np.shape(m_tot), dtype=np.float64)

        for j, _ in enumerate(cst):

            t = np.arange(cst[j], msa[j] + 1e6, 1e6)

            alpha[j] = (m_tot[j] - sfr[j] * cst[j]) / np.trapezoid(t * np.exp(-t / tau[j]), t)

        #print(alpha)

        t = np.arange(0, np.max(msa) + 1e6, 1e6)
        
        #print((np.shape(m_tot), np.shape(t)))

        curves = np.zeros((len(m_tot), len(t)))

        for j, _ in enumerate(curves):

            curves[j] = alpha[j] * t * np.exp(-t / tau[j])

        fig, ax = plt.subplots()

        medians, lowers, uppers = np.zeros(len(t)), np.zeros(len(t)), np.zeros(len(t))

        for j, _ in enumerate(t):

            median, lower, upper = weighted_quantile(curves[:,j], probs, [0.5, 0.16, 0.84])

            medians[j] = median
            lowers[j] = lower
            uppers[j] = upper

        #ax.plot(t / 1e6, np.median(curves, axis=0), c='black')
        ax.plot(t / 1e6, medians, c='black')
        ax.fill_between(t / 1e6, lowers, uppers, color='black', alpha=0.2)

        ax.set_xlabel('Lookback time (Myr)')
        ax.set_ylabel('Star formation rate (M$_\odot$ yr$^{-1}$)')

        ax.set_xlim(left=1)
        ax.set_ylim(bottom=0)

        ax.set_xscale('log')

        at = AnchoredText(id, loc='upper right', frameon=False, prop=dict(fontweight='bold'))
        ax.add_artist(at)

        # Make a scatter plot of the parameter and EWs for the same objects
        #dict_figs[fit][1].errorbar(me, ews[idx][0], xerr=[[me - me_lower], [me_upper - me]], yerr=[[ews[idx][0] - ews_lower[idx][0]], [ews_upper[idx][0] - ews[idx][0]]], color=dict_fits[fit][1], label=f'{dict_fits[fit][0] if l==0 else ''}', alpha=0.5)

In [ ]:
plot()